In [1]:
import pandas as pd
from sqlalchemy import create_engine

# ── 0. MySQL 접속 ────────────────────
# 실제 접속 정보로 아래 값을 수정하세요.
DB_USER = "dev"
DB_PASS = "pwd"
DB_HOST = "localhost"
DB_PORT = "3307"
DB_NAME = "orders"

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# ── 제외할 customerId 목록 ──────────────────────────────
excluded_ids = [
    '01H8K9XWK972V31SN3HDRGY022','01H7S6309C3ZWPVVMAWDYQDQE9',
    '01H7JEGZX7C0NCPN8ZZ3AJXASE','01H7YB33EPCWHTX1WWA9ANJCPG',
    '01H7VSQW5TB8Q4HR1Z4S5CTJKQ','01H4001EP0F1859987V7D4YJFG',
    '01H8DZZ71SYRKB4868B0WHV68E','01H98VZ8Y98MJ489TEJW7QCDS4',
    '01H7S0P3HK62VATNYDHADQPHKJ','01H9A1GTJVCTA13SA9YPEB9ETA',
    '01H7QY348FZQRVP21JAN8X4Z7R','01H7S9MDFE2FW13DPP93DV0WWP',
    '01H7SVW6ECGZ5JXPSV0YS6P7KG','01H7J0A52BKC867SHMNBZ85XTC',
    '01H847J2DGX5KEKMX386TKYGY1','01H7HNC3GKV65TAG2H3SGW7A0H',
    '01H98W8WA1X2E09TZ50DHHRJSA','01H59HRM9W7WZC0Y0Y5YK4ZJS8',
    '01H7HTY1JNFB3H0MP72X5R4QJW','01H7QXJZ7MQ63FTGJTJ5WB5TFR',
    '01H98VE7QSYH0W7AFH9Z9RY6KZ','01H3ZY9J55GM6S18YQGR3DCGQT',
    '01H8Y02HDQZTAH5AVKP51RN05G','01H847HNN73J8V4DN1V1Z7QJTT',
    '01GZZ0C5HVG9049C8AT5Y96HFS','01H9MSJKX4CCDBG2JK327BSP84',
    '01H9MAY8ZDX7KMFBS7EJYJR5AF','01HFDTAW1QRVR8P1HDKH4CNM47',
    '01HF8FYGWYH8Y27TMDPDAK2MH1','01HDJHZ6WM6QQAG627YQ549S8Z',
    '01HF3DR2VBV3D2M0SB6M32WXHQ'
]
excluded_ids_str = ', '.join(f"'{cid}'" for cid in excluded_ids)

# ── 1. 방문 단위 데이터 (세션별 최초 주문) ────────
query_visits = f"""
    SELECT
        customerId,
        sessionId,
        MIN(storeId)   AS storeId,
        MIN(createdAt) AS createdAt
    FROM orders
    WHERE customerId NOT IN ({excluded_ids_str})
    GROUP BY customerId, sessionId
"""
visits = pd.read_sql(query_visits, engine, parse_dates=["createdAt"])

# ── 2. 고객 × 매장 단위 방문 분석 ─────────────
def avg_interval(series):
    if len(series) < 2:
        return None
    s = series.sort_values()
    return (s.diff().dt.total_seconds() / 86_400).mean()

agg = (
    visits.groupby(["customerId", "storeId"])
          .agg(visit_count=('createdAt', 'size'),
               first_visit=('createdAt', 'min'),
               last_visit=('createdAt', 'max'),
               avg_revisit_days=('createdAt', avg_interval))
          .reset_index()
)

# ── 3. 주요 매장 선정 (방문 수 최다) ─────────────
top_idx = agg.groupby("customerId")["visit_count"].idxmax()
main_store = (
    agg.loc[top_idx]
       .rename(columns={"storeId": "main_storeId",
                        "visit_count": "main_store_visits",
                        "avg_revisit_days": "main_store_avg_revisit_days"})
)

# ── 4. 고객 요약 테이블 ────────────────────
summary = (
    visits.groupby("customerId")
          .size()
          .reset_index(name="total_visits")
          .merge(main_store, on="customerId")
          .sort_values("total_visits", ascending=False)
          .reset_index(drop=True)
)

# ── 5. 결과 확인 ────────────────────────
display(summary.head(50))

,customerId,total_visits,main_storeId,main_store_visits,first_visit,last_visit,main_store_avg_revisit_days
0,01GYNFXQNDD0ATWHDFXCBHD5XG,626,3,456,2023-08-10 11:03:30.813545,2023-09-20 10:55:08.505207,0.090097
1,01JMYTEJKA4YKPEYRZMHQ5CDXH,104,10011,79,2025-03-01 03:18:25.597374,2025-03-24 08:41:44.640538,0.297750
2,01HJSQYASPCWZ1P0ZM9JE85AV0,77,16,22,2024-01-03 02:59:56.078862,2024-05-02 02:30:37.493148,5.713316
3,01J4GEK224ASG6SP8Y6292G6GF,73,10029,53,2024-08-05 09:15:52.906374,2024-10-11 00:43:43.407217,1.281622
4,01JASB86MWHHQ6JBR7X7RNVTM5,64,10066,14,2025-05-02 06:51:41.505473,2025-05-02 08:53:25.162228,0.006503
5,01HMD870VA0J7MH5BZDYB6MDHX,63,10,26,2024-01-24 03:06:38.802240,2024-05-02 02:46:01.185533,3.959427
6,01JAQ39RK68Q5Z14Z3KF6HNMPR,63,10006,27,2024-10-21 08:44:24.685268,2025-02-04 00:45:44.537988,4.064138
7,01HMWHSV1Q4JV0M35CDAQM9WAN,60,16,21,2024-01-24 01:34:50.683867,2024-04-19 03:18:03.917205,4.303584
8,01HJT1EESSP7KAK3PZ1DZT85YS,58,3,58,2023-12-29 05:40:31.260502,2025-02-15 06:05:38.463367,7.263464
9,01HYCV37GAEZAMJJS6HT3N961X,52,3,28,2024-05-21 05:48:25.514666,2024-05-22 12:23:30.971184,0.047199


In [2]:
pd.set_option('display.max_rows', None)

In [3]:
summary[
    summary['main_storeId'] == 10028
].head(100)

,customerId,total_visits,main_storeId,main_store_visits,first_visit,last_visit,main_store_avg_revisit_days
11,01JR4YRCW1XC0QE1FFDPAQ2PS3,48,10028,48,2025-04-06 07:06:36.835488,2025-05-09 11:36:21.979392,0.706113
39,01J87GTZW3FZVEHTHEHWF62HBA,27,10028,27,2024-09-20 10:49:37.782954,2025-05-02 10:51:08.824606,8.615425
64,01JKSDCD52SAQ2BFTRDXFPX24F,22,10028,22,2025-02-11 02:28:32.182984,2025-04-24 08:41:30.175039,3.440905
70,01J48NVC08SYZ28MP51PVMC9YX,21,10028,21,2024-08-02 04:33:58.338397,2024-09-30 03:22:05.825727,2.947504
75,01J8EJ7D84PJB5CYZ15XDVFCSC,21,10028,20,2024-09-23 04:28:38.965876,2025-05-12 08:54:55.794928,12.167627
82,01J7CSDEPJ6ZAXJW1KPXKVBTCM,20,10028,20,2024-09-10 01:40:19.831744,2024-09-23 02:36:26.577614,0.686261
83,01J5274R9DXM751GGA8D3JMX8G,20,10028,20,2024-08-12 02:37:12.420777,2024-12-17 10:00:12.272725,6.700402
104,01JJR5VNGYXF49M5TSRQWDBATE,18,10028,16,2024-08-14 02:01:42.912445,2025-01-21 01:47:01.589289,10.665987
123,01JMNNE6KSFFEJEJ20S5Z4G4KG,16,10028,16,2025-02-22 01:48:03.664333,2025-05-10 09:28:45.197610,5.154662
127,01JAC1K3KWPVSHSHV951YJB2W3,16,10028,16,2024-10-17 01:30:31.572569,2024-12-20 04:24:36.417868,4.274726


In [4]:
# customers 테이블에서 필요한 컬럼만 불러오기
query_customers = """
    SELECT id AS customerId, fullName, phoneNumber, gender, dateOfBirth, subscribedToSicpamaKakaoChannel, isMarketingAgreed
    FROM customers
"""
customers = pd.read_sql(query_customers, engine)

# summary와 customerId 기준으로 join
customers = customers.merge(summary, on="customerId", how="right")

customers[
    (customers['main_storeId'] == 10028)
].head(100)

,customerId,fullName,phoneNumber,gender,dateOfBirth,subscribedToSicpamaKakaoChannel,isMarketingAgreed,total_visits,main_storeId,main_store_visits,first_visit,last_visit,main_store_avg_revisit_days
11,01JR4YRCW1XC0QE1FFDPAQ2PS3,G***t,None,None,None,0,NaT,48,10028,48,2025-04-06 07:06:36.835488,2025-05-09 11:36:21.979392,0.706113
39,01J87GTZW3FZVEHTHEHWF62HBA,하*우,************2624,male,1999-06-11,0,NaT,27,10028,27,2024-09-20 10:49:37.782954,2025-05-02 10:51:08.824606,8.615425
64,01JKSDCD52SAQ2BFTRDXFPX24F,G***t,None,None,None,0,NaT,22,10028,22,2025-02-11 02:28:32.182984,2025-04-24 08:41:30.175039,3.440905
70,01J48NVC08SYZ28MP51PVMC9YX,G***t,None,None,None,0,NaT,21,10028,21,2024-08-02 04:33:58.338397,2024-09-30 03:22:05.825727,2.947504
75,01J8EJ7D84PJB5CYZ15XDVFCSC,신*준,************6547,male,2001-08-30,0,NaT,21,10028,20,2024-09-23 04:28:38.965876,2025-05-12 08:54:55.794928,12.167627
82,01J7CSDEPJ6ZAXJW1KPXKVBTCM,G***t,**********6340,None,None,0,2024-09-10 01:43:51,20,10028,20,2024-09-10 01:40:19.831744,2024-09-23 02:36:26.577614,0.686261
83,01J5274R9DXM751GGA8D3JMX8G,G***t,**********3522,None,None,0,NaT,20,10028,20,2024-08-12 02:37:12.420777,2024-12-17 10:00:12.272725,6.700402
104,01JJR5VNGYXF49M5TSRQWDBATE,전*구,************7294,male,1985-04-25,1,NaT,18,10028,16,2024-08-14 02:01:42.912445,2025-01-21 01:47:01.589289,10.665987
123,01JMNNE6KSFFEJEJ20S5Z4G4KG,G***t,None,None,None,0,NaT,16,10028,16,2025-02-22 01:48:03.664333,2025-05-10 09:28:45.197610,5.154662
127,01JAC1K3KWPVSHSHV951YJB2W3,이*희,************4302,female,2005-05-06,1,NaT,16,10028,16,2024-10-17 01:30:31.572569,2024-12-20 04:24:36.417868,4.274726


In [14]:
customers[
    customers['customerId'] == '01JR4YRCW1XC0QE1FFDPAQ2PS3'
]

,customerId,fullName,phoneNumber,gender,dateOfBirth,subscribedToSicpamaKakaoChannel,isMarketingAgreed,total_visits,main_storeId,main_store_visits,first_visit,last_visit,main_store_avg_revisit_days
11,01JR4YRCW1XC0QE1FFDPAQ2PS3,G***t,None,None,None,0,NaT,48,10028,48,2025-04-06 07:06:36.835488,2025-05-09 11:36:21.979392,0.706113


In [6]:
c1 = customers[
    (customers['main_storeId'] == 10028) &
    ~customers['isMarketingAgreed'].isnull()
]
c1

,customerId,fullName,phoneNumber,gender,dateOfBirth,subscribedToSicpamaKakaoChannel,isMarketingAgreed,total_visits,main_storeId,main_store_visits,first_visit,last_visit,main_store_avg_revisit_days
82,01J7CSDEPJ6ZAXJW1KPXKVBTCM,G***t,**********6340,None,None,0,2024-09-10 01:43:51,20,10028,20,2024-09-10 01:40:19.831744,2024-09-23 02:36:26.577614,0.686261
222,01JAY33R5XQ9A6ZXNY9WF2P81X,이*빈,************3373,female,2005-12-22,0,2024-11-06 03:36:11,12,10028,8,2024-10-24 01:44:11.002804,2025-03-21 13:05:13.195023,21.210420
481,01J7ASSQ9FWBHBWXWEEZV8FCGN,이*윤,************1871,male,1995-03-01,1,2024-09-09 07:10:34,8,10028,8,2024-09-09 07:08:13.770389,2024-09-22 11:02:11.682580,1.880354
837,01JAQ9HMDBTCKJ9Z95HZ5RHTH7,김*슬,************4524,female,2005-04-24,1,2024-10-21 10:25:13,6,10028,6,2024-10-21 10:21:29.612738,2024-12-14 06:59:20.669379,10.771924
849,01J7JQ5ND43HZ2B6DJQAF4AJ58,G***t,**********5077,None,None,0,2024-09-12 08:57:36,6,10028,6,2024-09-12 08:56:33.987678,2024-12-14 12:20:54.780512,18.628381
915,01J7NDA8R08KMZRTAQ7GAEQ77D,김*우,************6784,male,1998-01-22,1,2024-09-13 10:06:27,6,10028,6,2024-09-13 10:01:48.756808,2024-09-23 05:16:56.313646,1.960434
964,01J9K3VPW2ENWAJ1B410GCDBGF,박*원,************4919,female,2002-05-30,1,2024-10-07 09:10:48,6,10028,6,2024-10-07 09:09:19.781079,2024-11-04 07:28:59.667266,5.586065
1144,01J4ZZSVJ6HDH8F4FJTJ7PHP85,G***t,**********6709,None,None,0,2024-09-09 03:12:57,5,10028,5,2024-08-11 05:50:26.417109,2024-09-09 03:11:24.497607,7.222390
1378,01JA9QXTN06K5H0GZ7NGPMYNXE,김*준,************3990,male,2004-06-29,1,2024-10-16 04:07:49,5,10028,5,2024-10-16 04:03:26.682643,2024-10-20 04:30:38.713605,1.004722
1453,01J9V12JYSJCD32XP655TEWSZX,윤*수,************6883,male,1999-02-09,1,2024-10-10 10:56:14,5,10028,5,2024-10-10 10:54:37.777245,2024-11-04 11:00:19.177961,6.250988


In [10]:
customers[
    ~customers['isMarketingAgreed'].isnull()
].info()

<class 'pandas.core.frame.DataFrame'>
Index: 870 entries, 35 to 66386
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   customerId                       870 non-null    object        
 1   fullName                         870 non-null    object        
 2   phoneNumber                      870 non-null    object        
 3   gender                           534 non-null    object        
 4   dateOfBirth                      535 non-null    object        
 5   subscribedToSicpamaKakaoChannel  870 non-null    int64         
 6   isMarketingAgreed                870 non-null    datetime64[ns]
 7   total_visits                     870 non-null    int64         
 8   main_storeId                     870 non-null    int64         
 9   main_store_visits                870 non-null    int64         
 10  first_visit                      870 non-null    datetime64[ns]


In [13]:
customers[
    ~customers['isMarketingAgreed'].isnull() &
    customers['gender'].isnull()
]

,customerId,fullName,phoneNumber,gender,dateOfBirth,subscribedToSicpamaKakaoChannel,isMarketingAgreed,total_visits,main_storeId,main_store_visits,first_visit,last_visit,main_store_avg_revisit_days
82,01J7CSDEPJ6ZAXJW1KPXKVBTCM,G***t,**********6340,None,None,0,2024-09-10 01:43:51,20,10028,20,2024-09-10 01:40:19.831744,2024-09-23 02:36:26.577614,0.686261
175,01JG10GRAC7PQX50PR85FYM93H,G***t,**********7150,None,None,0,2024-12-26 08:18:11,13,3,12,2024-12-26 08:14:51.258634,2025-01-04 02:46:28.147817,0.797450
315,01J7DJGX9PXSGASRGTF112XPBC,G***t,**********3312,None,None,0,2024-09-22 08:09:04,10,10023,10,2024-09-10 08:59:07.474598,2024-10-21 09:44:08.015721,4.559028
505,01J6KJEMNWVFRTMWBMYJB5KEYZ,홍*진,**********0499,None,None,0,2024-08-31 06:39:54,8,10026,6,2024-08-31 06:37:17.565604,2024-09-02 08:40:01.597653,0.417046
849,01J7JQ5ND43HZ2B6DJQAF4AJ58,G***t,**********5077,None,None,0,2024-09-12 08:57:36,6,10028,6,2024-09-12 08:56:33.987678,2024-12-14 12:20:54.780512,18.628381
1144,01J4ZZSVJ6HDH8F4FJTJ7PHP85,G***t,**********6709,None,None,0,2024-09-09 03:12:57,5,10028,5,2024-08-11 05:50:26.417109,2024-09-09 03:11:24.497607,7.222390
1330,01JAHJN0Q2AAB4CXX2HKPYYZFW,G***t,*********5135,None,None,0,2024-10-19 05:14:13,5,10023,5,2024-10-19 05:05:25.261009,2024-10-19 09:39:01.049861,0.047499
1521,01JTJJKWXD5EVFFARPAMGJF4YX,G***t,**********0116,None,None,0,2025-05-06 10:38:49,4,3,4,2025-05-06 10:35:12.201443,2025-05-12 12:50:10.722004,2.031244
1590,01JFY7KMYN4D9R8QNX2PSCN2QF,G***t,**********1503,None,None,0,2024-12-25 06:22:29,4,3,4,2024-12-25 06:21:02.860017,2024-12-26 12:35:33.090615,0.420024
1698,01JSKDV23WE7B3B2YMN6DQT88G,G***t,**********0111,None,None,0,2025-05-07 04:36:06,4,5,4,2025-04-24 08:15:16.626777,2025-05-10 08:52:11.607949,5.341879


In [12]:
customers[
    ~customers['isMarketingAgreed'].isnull()
].groupby('main_storeId').count()

,customerId,fullName,phoneNumber,gender,dateOfBirth,subscribedToSicpamaKakaoChannel,isMarketingAgreed,total_visits,main_store_visits,first_visit,last_visit,main_store_avg_revisit_days
main_storeId,,,,,,,,,,,,
3,466,466,466,281,281,466,466,466,466,466,466,190
5,33,33,33,28,28,33,33,33,33,33,33,20
15,19,19,19,12,12,19,19,19,19,19,19,10
10000,2,2,2,1,1,2,2,2,2,2,2,0
10005,2,2,2,1,1,2,2,2,2,2,2,0
10023,122,122,122,90,90,122,122,122,122,122,122,55
10024,4,4,4,1,1,4,4,4,4,4,4,1
10026,6,6,6,3,3,6,6,6,6,6,6,4
10028,212,212,212,114,115,212,212,212,212,212,212,87


In [8]:
c1.to_csv('10028.csv', index=False)